<a href="https://colab.research.google.com/github/santiagonajera/MODELACION-Y-PRONOSTICOS-DE-LA-DEMANDA/blob/main/Clasificador.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import requests
from io import BytesIO

print("="*80)
print("CLASIFICACIÓN COMPLETA: ABC-XYZ-ADI-CV²")
print("="*80)

# --- Paso 1: Descargar y leer datos ---
url = "https://github.com/santiagonajera/MODELACION-Y-PRONOSTICOS-DE-LA-DEMANDA/raw/refs/heads/main/EjercicioIntegral-Forecast.xlsx"
response = requests.get(url)
excel_file = BytesIO(response.content)

df_datos = pd.read_excel(excel_file, sheet_name='Datos')

# Limpiar nombres de columnas
df_datos.columns = df_datos.columns.astype(str).str.strip()
df_datos.rename(columns={df_datos.columns[0]: 'Producto'}, inplace=True)

# --- Paso 2: Buscar columna 'oct-25' ---
all_cols = df_datos.columns.tolist()
period_cols = [col for col in all_cols if col != 'Producto']

def find_oct25_column(columns):
    """Busca la columna de octubre 2025"""
    for col in columns:
        if str(col).lower().strip() == 'oct-25':
            return col

    variants = ['oct-25', 'Oct-25', 'oct 25', 'Oct 25', 'oct25', 'Oct25']
    for variant in variants:
        if variant in columns:
            return variant

    for col in columns:
        col_str = str(col).lower()
        if '2025' in col_str and '10' in col_str:
            return col

    oct_cols = [col for col in columns if 'oct' in str(col).lower() and '25' in str(col)]
    if oct_cols:
        return oct_cols[0]

    return None

target_col = find_oct25_column(period_cols)
if target_col is None:
    raise ValueError("No se encontró la columna 'oct-25'")

idx_target = period_cols.index(target_col)
cols_to_use = period_cols[:idx_target + 1]

print(f"\n✅ Periodo: {cols_to_use[0]} hasta {cols_to_use[-1]} ({len(cols_to_use)} períodos)")

# --- Paso 3: Preparar datos ---
ventas = df_datos[['Producto'] + cols_to_use].copy()
ventas.set_index('Producto', inplace=True)

# Convertir a numérico
for col in ventas.columns:
    ventas[col] = pd.to_numeric(ventas[col], errors='coerce')

ventas = ventas.dropna(how='all')

print(f"📊 Productos analizados: {len(ventas)}")

# --- Paso 4: Clasificación ABC ---
print("\n" + "="*80)
print("PASO 1: CLASIFICACIÓN ABC")
print("="*80)

# Calcular ventas totales
ventas_totales = ventas.sum(axis=1).sort_values(ascending=False)
ventas_totales_pct = (ventas_totales / ventas_totales.sum() * 100)
ventas_acum_pct = ventas_totales_pct.cumsum()

# Clasificar ABC
def clasificar_abc(pct_acum):
    if pct_acum <= 80:
        return 'A'
    elif pct_acum <= 95:
        return 'B'
    else:
        return 'C'

abc_class = ventas_acum_pct.apply(clasificar_abc)

print(f"\n📊 Clasificación ABC (80-15-5):")
print(abc_class.value_counts().sort_index())

# --- Paso 5: Clasificación XYZ (CV) ---
print("\n" + "="*80)
print("PASO 2: CLASIFICACIÓN XYZ (Coeficiente de Variación)")
print("="*80)

mean = ventas.mean(axis=1)
std = ventas.std(axis=1)
cv = (std / mean).replace([float('inf'), -float('inf')], pd.NA)

# Clasificar XYZ por percentiles
p33 = cv.quantile(0.33)
p67 = cv.quantile(0.67)

def xyz_class(cv_val):
    if pd.isna(cv_val):
        return None
    if cv_val <= p33:
        return 'X'
    elif cv_val <= p67:
        return 'Y'
    else:
        return 'Z'

xyz = cv.apply(xyz_class)

print(f"\n📊 Clasificación XYZ:")
print(xyz.value_counts().sort_index())
print(f"   P33 = {p33:.4f}")
print(f"   P67 = {p67:.4f}")

# --- Paso 6: Calcular ADI para items Z ---
print("\n" + "="*80)
print("PASO 3: CÁLCULO DE ADI (Average Demand Interval) para items Z")
print("="*80)

def calcular_adi(serie):
    """
    ADI = Total períodos / Períodos con demanda > 0
    ADI < 1.32: Continua
    ADI >= 1.32: Intermitente
    """
    total_periodos = len(serie)
    periodos_con_demanda = (serie > 0).sum()

    if periodos_con_demanda == 0:
        return np.nan

    return total_periodos / periodos_con_demanda

# Calcular ADI para todos
adi = ventas.apply(calcular_adi, axis=1)

# Clasificar tipo de demanda
def tipo_demanda(row):
    if row['XYZ'] in ['X', 'Y']:
        return 'Continua'
    else:  # Z
        if pd.isna(row['ADI']):
            return 'Sin datos'
        elif row['ADI'] < 1.32:
            return 'Continua'
        else:
            return 'Intermitente'

# Crear DataFrame maestro
df_master = pd.DataFrame({
    'Producto': ventas.index,
    'Ventas_Total': ventas_totales,
    'ABC': abc_class,
    'Media': mean,
    'Desv_Std': std,
    'CV': cv,
    'XYZ': xyz,
    'ADI': adi
})

df_master['Tipo_Demanda'] = df_master.apply(tipo_demanda, axis=1)

print(f"\n📊 Tipo de Demanda:")
print(df_master['Tipo_Demanda'].value_counts())

# --- Paso 7: Calcular CV² para items intermitentes ---
print("\n" + "="*80)
print("PASO 4: CÁLCULO DE CV² para items INTERMITENTES")
print("="*80)

print("""
📖 Clasificación según CV² (Syntetos et al.):
   • CV² < 0.49: Suave (Smooth) - Forecasteable con Croston/SBA
   • CV² >= 0.49: Errática (Erratic) - Muy difícil de forecastear
""")

# Calcular CV² para todos (aunque solo nos interesan los intermitentes)
df_master['CV2'] = df_master['CV'] ** 2

# Clasificar según CV² (solo para intermitentes)
def clasificar_cv2(row):
    if row['Tipo_Demanda'] == 'Intermitente':
        if pd.isna(row['CV2']):
            return 'Sin datos'
        elif row['CV2'] < 0.49:
            return 'Suave (Forecasteable)'
        else:
            return 'Errática (Difícil)'
    else:
        return 'N/A'

df_master['CV2_Clasificacion'] = df_master.apply(clasificar_cv2, axis=1)

# Mostrar estadísticas de items intermitentes
items_intermitentes = df_master[df_master['Tipo_Demanda'] == 'Intermitente']

if len(items_intermitentes) > 0:
    print(f"\n📊 Total items intermitentes: {len(items_intermitentes)}")
    print(f"\n   Estadísticas de CV²:")
    print(f"   • Media: {items_intermitentes['CV2'].mean():.4f}")
    print(f"   • Mediana: {items_intermitentes['CV2'].median():.4f}")
    print(f"   • Mínimo: {items_intermitentes['CV2'].min():.4f}")
    print(f"   • Máximo: {items_intermitentes['CV2'].max():.4f}")

    print(f"\n📊 Clasificación por CV²:")
    cv2_counts = df_master[df_master['Tipo_Demanda'] == 'Intermitente']['CV2_Clasificacion'].value_counts()
    print(cv2_counts)

    print(f"\n🔍 Detalle de items intermitentes:")
    print(items_intermitentes[['Producto', 'ABC', 'CV', 'CV2', 'ADI', 'CV2_Clasificacion']].sort_values('CV2', ascending=False).to_string(index=False))
else:
    print("⚠️  No hay items intermitentes en este dataset")

# --- Paso 8: Clasificación Final Detallada ---
print("\n" + "="*80)
print("PASO 5: CLASIFICACIÓN FINAL DETALLADA")
print("="*80)

def clasificacion_final_detallada(row):
    """
    Genera la clasificación final completa
    """
    abc = row['ABC']
    xyz = row['XYZ']
    tipo = row['Tipo_Demanda']

    if tipo == 'Continua':
        return f"{abc}{xyz}-Continua"
    elif tipo == 'Intermitente':
        cv2_clase = row['CV2_Clasificacion']
        if 'Suave' in cv2_clase:
            return f"{abc}{xyz}-Intermitente-Suave"
        elif 'Errática' in cv2_clase:
            return f"{abc}{xyz}-Intermitente-Errática"
        else:
            return f"{abc}{xyz}-Intermitente-{cv2_clase}"
    else:
        return f"{abc}{xyz}-{tipo}"

df_master['Clasificacion_Final'] = df_master.apply(clasificacion_final_detallada, axis=1)

# --- Paso 9: GENERAR DATAFRAME RESUMEN SOLICITADO ---
print("\n" + "="*80)
print("📊 DATAFRAME RESUMEN: CONTEO POR COMBINACIÓN ABC-XYZ-TIPO")
print("="*80)

# Crear DataFrame de resumen con todas las combinaciones
resumen = df_master.groupby(['ABC', 'XYZ', 'Tipo_Demanda', 'CV2_Clasificacion']).size().reset_index(name='Cantidad')

# Crear columna de clasificación simplificada
resumen['Clasificacion'] = resumen.apply(
    lambda row: f"{row['ABC']}{row['XYZ']}-" +
                (f"{row['Tipo_Demanda']}" if row['Tipo_Demanda'] == 'Continua'
                 else f"Intermitente ({row['CV2_Clasificacion']})"),
    axis=1
)

# Ordenar
resumen = resumen.sort_values(['ABC', 'XYZ', 'Tipo_Demanda'])

print("\n📋 RESUMEN DETALLADO:")
print(resumen.to_string(index=False))

# Crear DataFrame resumen simplificado
print("\n" + "="*80)
print("📊 RESUMEN SIMPLIFICADO (Agrupado)")
print("="*80)

# Agrupar por ABC-XYZ-Tipo (sin distinguir CV2 para continuas)
resumen_simple = df_master.groupby(['ABC', 'XYZ', 'Tipo_Demanda']).agg({
    'Producto': 'count',
    'Ventas_Total': 'sum',
    'CV': 'mean',
    'ADI': 'mean',
    'CV2': 'mean'
}).reset_index()

resumen_simple.columns = ['ABC', 'XYZ', 'Tipo_Demanda', 'Cantidad', 'Ventas_Total', 'CV_Promedio', 'ADI_Promedio', 'CV2_Promedio']

# Crear columna combinada
resumen_simple['Grupo'] = resumen_simple['ABC'] + resumen_simple['XYZ'] + '-' + resumen_simple['Tipo_Demanda']

# Redondear valores
resumen_simple['CV_Promedio'] = resumen_simple['CV_Promedio'].round(4)
resumen_simple['ADI_Promedio'] = resumen_simple['ADI_Promedio'].round(3)
resumen_simple['CV2_Promedio'] = resumen_simple['CV2_Promedio'].round(4)
resumen_simple['Ventas_Total'] = resumen_simple['Ventas_Total'].round(0)

print("\n" + resumen_simple.to_string(index=False))

# --- Paso 10: Resumen por combinación ABC-XYZ-TIPO ---
print("\n" + "="*80)
print("📊 TABLA FINAL: CANTIDAD POR GRUPO")
print("="*80)

# Crear tabla pivote
tabla_final = resumen_simple[['Grupo', 'Cantidad', 'CV_Promedio', 'ADI_Promedio']].copy()
tabla_final = tabla_final.sort_values('Cantidad', ascending=False)

print("\n" + tabla_final.to_string(index=False))

# Calcular totales y porcentajes
total_items = len(df_master)

print("\n" + "="*80)
print("📊 DISTRIBUCIÓN PORCENTUAL")
print("="*80)

tabla_pct = tabla_final.copy()
tabla_pct['Porcentaje'] = (tabla_pct['Cantidad'] / total_items * 100).round(2)
tabla_pct['Acumulado'] = tabla_pct['Porcentaje'].cumsum().round(2)

print("\n" + tabla_pct.to_string(index=False))

# --- Paso 11: Análisis especial de intermitentes ---
if len(items_intermitentes) > 0:
    print("\n" + "="*80)
    print("🚨 ANÁLISIS DETALLADO DE ITEMS INTERMITENTES")
    print("="*80)

    # Resumen por ABC para intermitentes
    intermitentes_abc = items_intermitentes.groupby(['ABC', 'CV2_Clasificacion']).size().reset_index(name='Cantidad')
    print("\n📊 Items Intermitentes por clase ABC y CV²:")
    print(intermitentes_abc.to_string(index=False))

    # Estadísticas por tipo
    print("\n📊 Estadísticas de Items Intermitentes:")

    suaves = items_intermitentes[items_intermitentes['CV2_Clasificacion'] == 'Suave (Forecasteable)']
    erraticas = items_intermitentes[items_intermitentes['CV2_Clasificacion'] == 'Errática (Difícil)']

    if len(suaves) > 0:
        print(f"\n   🟢 Intermitentes SUAVES (CV² < 0.49): {len(suaves)} items")
        print(f"      • Representan: {len(suaves)/len(items_intermitentes)*100:.1f}% de intermitentes")
        print(f"      • CV² promedio: {suaves['CV2'].mean():.4f}")
        print(f"      • ADI promedio: {suaves['ADI'].mean():.3f}")
        print(f"      • Método sugerido: Croston, SBA, TSB")

    if len(erraticas) > 0:
        print(f"\n   🔴 Intermitentes ERRÁTICAS (CV² >= 0.49): {len(erraticas)} items")
        print(f"      • Representan: {len(erraticas)/len(items_intermitentes)*100:.1f}% de intermitentes")
        print(f"      • CV² promedio: {erraticas['CV2'].mean():.4f}")
        print(f"      • ADI promedio: {erraticas['ADI'].mean():.3f}")
        print(f"      • Método sugerido: Bootstrap, Simulación, Gestión por excepción")

# --- Paso 12: Matriz de decisión ---
print("\n" + "="*80)
print("🎯 MATRIZ DE DECISIÓN Y ESTRATEGIAS")
print("="*80)

matriz_decision = []

for _, row in resumen_simple.iterrows():
    grupo = row['Grupo']
    cant = row['Cantidad']
    abc = row['ABC']
    xyz = row['XYZ']
    tipo = row['Tipo_Demanda']

    # Determinar prioridad
    if abc == 'A':
        prioridad = 'ALTA'
    elif abc == 'B':
        prioridad = 'MEDIA'
    else:
        prioridad = 'BAJA'

    # Determinar método de pronóstico
    if tipo == 'Continua':
        if xyz == 'X':
            metodo = 'Promedio Móvil, SES'
        elif xyz == 'Y':
            metodo = 'Holt, Winter\'s'
        else:  # Z
            metodo = 'ARIMA, ML'
    else:  # Intermitente
        cv2_prom = row['CV2_Promedio']
        if pd.notna(cv2_prom) and cv2_prom < 0.49:
            metodo = 'Croston, SBA'
        else:
            metodo = 'TSB, Bootstrap'

    # Determinar estrategia de inventario
    if tipo == 'Continua':
        if xyz == 'X':
            inventario = 'Stock bajo, EOQ'
        elif xyz == 'Y':
            inventario = 'Stock medio, (s,Q)'
        else:
            inventario = 'Stock alto, revisión frecuente'
    else:
        if abc == 'A':
            inventario = 'Stock estratégico, monitoreo diario'
        else:
            inventario = 'Evaluar bajo pedido'

    matriz_decision.append({
        'Grupo': grupo,
        'Cantidad': cant,
        'Prioridad': prioridad,
        'Método_Pronóstico': metodo,
        'Estrategia_Inventario': inventario
    })

df_decision = pd.DataFrame(matriz_decision)
print("\n" + df_decision.to_string(index=False))

# --- Paso 13: Resumen ejecutivo ---
print("\n" + "="*80)
print("📊 RESUMEN EJECUTIVO")
print("="*80)

total_continua = len(df_master[df_master['Tipo_Demanda'] == 'Continua'])
total_intermitente = len(df_master[df_master['Tipo_Demanda'] == 'Intermitente'])

print(f"""
Total de items analizados: {len(df_master)}

CLASIFICACIÓN ABC:
  • Clase A: {len(df_master[df_master['ABC'] == 'A'])} items ({len(df_master[df_master['ABC'] == 'A'])/len(df_master)*100:.1f}%)
  • Clase B: {len(df_master[df_master['ABC'] == 'B'])} items ({len(df_master[df_master['ABC'] == 'B'])/len(df_master)*100:.1f}%)
  • Clase C: {len(df_master[df_master['ABC'] == 'C'])} items ({len(df_master[df_master['ABC'] == 'C'])/len(df_master)*100:.1f}%)

CLASIFICACIÓN XYZ:
  • Clase X: {len(df_master[df_master['XYZ'] == 'X'])} items ({len(df_master[df_master['XYZ'] == 'X'])/len(df_master)*100:.1f}%)
  • Clase Y: {len(df_master[df_master['XYZ'] == 'Y'])} items ({len(df_master[df_master['XYZ'] == 'Y'])/len(df_master)*100:.1f}%)
  • Clase Z: {len(df_master[df_master['XYZ'] == 'Z'])} items ({len(df_master[df_master['XYZ'] == 'Z'])/len(df_master)*100:.1f}%)

TIPO DE DEMANDA:
  • Continua: {total_continua} items ({total_continua/len(df_master)*100:.1f}%)
  • Intermitente: {total_intermitente} items ({total_intermitente/len(df_master)*100:.1f}%)
""")

if total_intermitente > 0:
    suaves_count = len(df_master[df_master['CV2_Clasificacion'] == 'Suave (Forecasteable)'])
    erraticas_count = len(df_master[df_master['CV2_Clasificacion'] == 'Errática (Difícil)'])

    print(f"""
ITEMS INTERMITENTES (Detalle):
  • Suaves (CV² < 0.49): {suaves_count} items ({suaves_count/total_intermitente*100:.1f}% de intermitentes)
    → Forecasteables con Croston/SBA
  • Erráticas (CV² >= 0.49): {erraticas_count} items ({erraticas_count/total_intermitente*100:.1f}% de intermitentes)
    → Muy difíciles, requieren métodos especiales
""")

# Items críticos (A-Intermitente)
criticos = df_master[(df_master['ABC'] == 'A') & (df_master['Tipo_Demanda'] == 'Intermitente')]
if len(criticos) > 0:
    print(f"""
⚠️  ITEMS CRÍTICOS (A-Intermitente): {len(criticos)} items
    → Requieren ATENCIÓN ESPECIAL
    → Alta importancia + Alta complejidad
""")
    print("   Productos críticos:")
    for producto in criticos['Producto'].values[:10]:
        print(f"   • {producto}")
    if len(criticos) > 10:
        print(f"   ... y {len(criticos)-10} más")

print("\n" + "="*80)
print("✅ ANÁLISIS COMPLETADO")
print("="*80)

# Retornar DataFrames principales
print("\n💡 DataFrames generados en memoria:")
print("   • df_master: Clasificación completa de todos los items")
print("   • resumen_simple: Resumen por grupo ABC-XYZ-Tipo")
print("   • tabla_pct: Tabla con porcentajes")
print("   • df_decision: Matriz de decisión con estrategias")

CLASIFICACIÓN COMPLETA: ABC-XYZ-ADI-CV²

✅ Periodo: 2023-01-01 00:00:00 hasta 2025-10-01 00:00:00 (34 períodos)
📊 Productos analizados: 30

PASO 1: CLASIFICACIÓN ABC

📊 Clasificación ABC (80-15-5):
A    22
B     4
C     4
Name: count, dtype: int64

PASO 2: CLASIFICACIÓN XYZ (Coeficiente de Variación)

📊 Clasificación XYZ:
X    10
Y    10
Z    10
Name: count, dtype: int64
   P33 = 0.5661
   P67 = 0.6377

PASO 3: CÁLCULO DE ADI (Average Demand Interval) para items Z

📊 Tipo de Demanda:
Tipo_Demanda
Continua        27
Intermitente     3
Name: count, dtype: int64

PASO 4: CÁLCULO DE CV² para items INTERMITENTES

📖 Clasificación según CV² (Syntetos et al.):
   • CV² < 0.49: Suave (Smooth) - Forecasteable con Croston/SBA
   • CV² >= 0.49: Errática (Erratic) - Muy difícil de forecastear


📊 Total items intermitentes: 3

   Estadísticas de CV²:
   • Media: 2.3021
   • Mediana: 2.2581
   • Mínimo: 1.9650
   • Máximo: 2.6831

📊 Clasificación por CV²:
CV2_Clasificacion
Errática (Difícil)    3
Nam

In [ ]:
import pandas as pd
import numpy as np
import requests
from io import BytesIO

print("="*80)
print("CLASIFICACIÓN COMPLETA: ABC-XYZ-ADI-CV²")
print("="*80)

# --- Paso 1: Descargar y leer datos ---
url = "https://github.com/santiagonajera/MODELACION-Y-PRONOSTICOS-DE-LA-DEMANDA/raw/refs/heads/main/EjercicioIntegral-Forecast.xlsx"
response = requests.get(url)
excel_file = BytesIO(response.content)

# CAMBIO: Usar la hoja 'Datos por Ejercicio' en lugar de 'Datos'
df_datos = pd.read_excel(excel_file, sheet_name='Ejercicio')

# Limpiar nombres de columnas
df_datos.columns = df_datos.columns.astype(str).str.strip()
df_datos.rename(columns={df_datos.columns[0]: 'Producto'}, inplace=True)

# --- Paso 2: Buscar columna 'oct-25' ---
all_cols = df_datos.columns.tolist()
period_cols = [col for col in all_cols if col != 'Producto']

def find_oct25_column(columns):
    """Busca la columna de octubre 2025"""
    for col in columns:
        if str(col).lower().strip() == 'oct-25':
            return col

    variants = ['oct-25', 'Oct-25', 'oct 25', 'Oct 25', 'oct25', 'Oct25']
    for variant in variants:
        if variant in columns:
            return variant

    for col in columns:
        col_str = str(col).lower()
        if '2025' in col_str and '10' in col_str:
            return col

    oct_cols = [col for col in columns if 'oct' in str(col).lower() and '25' in str(col)]
    if oct_cols:
        return oct_cols[0]

    return None

target_col = find_oct25_column(period_cols)
if target_col is None:
    raise ValueError("No se encontró la columna 'oct-25'")

idx_target = period_cols.index(target_col)
cols_to_use = period_cols[:idx_target + 1]

print(f"\n✅ Periodo: {cols_to_use[0]} hasta {cols_to_use[-1]} ({len(cols_to_use)} períodos)")

# --- Paso 3: Preparar datos ---
ventas = df_datos[['Producto'] + cols_to_use].copy()
ventas.set_index('Producto', inplace=True)

# Convertir a numérico
for col in ventas.columns:
    ventas[col] = pd.to_numeric(ventas[col], errors='coerce')

ventas = ventas.dropna(how='all')

print(f"📊 Productos analizados: {len(ventas)}")

# --- Paso 4: Clasificación ABC ---
print("\n" + "="*80)
print("PASO 1: CLASIFICACIÓN ABC")
print("="*80)

# Calcular ventas totales
ventas_totales = ventas.sum(axis=1).sort_values(ascending=False)
ventas_totales_pct = (ventas_totales / ventas_totales.sum() * 100)
ventas_acum_pct = ventas_totales_pct.cumsum()

# Clasificar ABC
def clasificar_abc(pct_acum):
    if pct_acum <= 80:
        return 'A'
    elif pct_acum <= 95:
        return 'B'
    else:
        return 'C'

abc_class = ventas_acum_pct.apply(clasificar_abc)

print(f"\n📊 Clasificación ABC (80-15-5):")
print(abc_class.value_counts().sort_index())

# --- Paso 5: Clasificación XYZ (CV) ---
print("\n" + "="*80)
print("PASO 2: CLASIFICACIÓN XYZ (Coeficiente de Variación)")
print("="*80)

mean = ventas.mean(axis=1)
std = ventas.std(axis=1)
cv = (std / mean).replace([float('inf'), -float('inf')], pd.NA)

# Clasificar XYZ por percentiles
p33 = cv.quantile(0.33)
p67 = cv.quantile(0.67)

def xyz_class(cv_val):
    if pd.isna(cv_val):
        return None
    if cv_val <= p33:
        return 'X'
    elif cv_val <= p67:
        return 'Y'
    else:
        return 'Z'

xyz = cv.apply(xyz_class)

print(f"\n📊 Clasificación XYZ:")
print(xyz.value_counts().sort_index())
print(f" P33 = {p33:.4f}")
print(f" P67 = {p67:.4f}")

# --- Paso 6: Calcular ADI para items Z ---
print("\n" + "="*80)
print("PASO 3: CÁLCULO DE ADI (Average Demand Interval) para items Z")
print("="*80)

def calcular_adi(serie):
    """
    ADI = Total períodos / Períodos con demanda > 0
    ADI < 1.32: Continua
    ADI >= 1.32: Intermitente
    """
    total_periodos = len(serie)
    periodos_con_demanda = (serie > 0).sum()

    if periodos_con_demanda == 0:
        return np.nan

    return total_periodos / periodos_con_demanda

# Calcular ADI para todos
adi = ventas.apply(calcular_adi, axis=1)

# Clasificar tipo de demanda
def tipo_demanda(row):
    if row['XYZ'] in ['X', 'Y']:
        return 'Continua'
    else: # Z
        if pd.isna(row['ADI']):
            return 'Sin datos'
        elif row['ADI'] < 1.32:
            return 'Continua'
        else:
            return 'Intermitente'

# Crear DataFrame maestro
df_master = pd.DataFrame({
    'Producto': ventas.index,
    'Ventas_Total': ventas_totales,
    'ABC': abc_class,
    'Media': mean,
    'Desv_Std': std,
    'CV': cv,
    'XYZ': xyz,
    'ADI': adi
})

df_master['Tipo_Demanda'] = df_master.apply(tipo_demanda, axis=1)

print(f"\n📊 Tipo de Demanda:")
print(df_master['Tipo_Demanda'].value_counts())

# --- Paso 7: Calcular CV² para items intermitentes ---
print("\n" + "="*80)
print("PASO 4: CÁLCULO DE CV² para items INTERMITENTES")
print("="*80)

print("""
📖 Clasificación según CV² (Syntetos et al.):
• CV² < 0.49: Suave (Smooth) - Forecasteable con Croston/SBA
• CV² >= 0.49: Errática (Erratic) - Muy difícil de forecastear
""")

# Calcular CV² para todos (aunque solo nos interesan los intermitentes)
df_master['CV2'] = df_master['CV'] ** 2

# Clasificar según CV² (solo para intermitentes)
def clasificar_cv2(row):
    if row['Tipo_Demanda'] == 'Intermitente':
        if pd.isna(row['CV2']):
            return 'Sin datos'
        elif row['CV2'] < 0.49:
            return 'Suave (Forecasteable)'
        else:
            return 'Errática (Difícil)'
    else:
        return 'N/A'

df_master['CV2_Clasificacion'] = df_master.apply(clasificar_cv2, axis=1)

# Mostrar estadísticas de items intermitentes
items_intermitentes = df_master[df_master['Tipo_Demanda'] == 'Intermitente']

if len(items_intermitentes) > 0:
    print(f"\n📊 Total items intermitentes: {len(items_intermitentes)}")
    print(f"\n Estadísticas de CV²:")
    print(f" • Media: {items_intermitentes['CV2'].mean():.4f}")
    print(f" • Mediana: {items_intermitentes['CV2'].median():.4f}")
    print(f" • Mínimo: {items_intermitentes['CV2'].min():.4f}")
    print(f" • Máximo: {items_intermitentes['CV2'].max():.4f}")

    print(f"\n📊 Clasificación por CV²:")
    cv2_counts = df_master[df_master['Tipo_Demanda'] == 'Intermitente']['CV2_Clasificacion'].value_counts()
    print(cv2_counts)

    print(f"\n🔍 Detalle de items intermitentes:")
    print(items_intermitentes[['Producto', 'ABC', 'CV', 'CV2', 'ADI', 'CV2_Clasificacion']].sort_values('CV2', ascending=False).to_string(index=False))
else:
    print("⚠️ No hay items intermitentes en este dataset")

# --- Paso 8: Clasificación Final Detallada ---
print("\n" + "="*80)
print("PASO 5: CLASIFICACIÓN FINAL DETALLADA")
print("="*80)

def clasificacion_final_detallada(row):
    """
    Genera la clasificación final completa
    """
    abc = row['ABC']
    xyz = row['XYZ']
    tipo = row['Tipo_Demanda']

    if tipo == 'Continua':
        return f"{abc}{xyz}-Continua"
    elif tipo == 'Intermitente':
        cv2_clase = row['CV2_Clasificacion']
        if 'Suave' in cv2_clase:
            return f"{abc}{xyz}-Intermitente-Suave"
        elif 'Errática' in cv2_clase:
            return f"{abc}{xyz}-Intermitente-Errática"
        else:
            return f"{abc}{xyz}-Intermitente-{cv2_clase}"
    else:
        return f"{abc}{xyz}-{tipo}"

df_master['Clasificacion_Final'] = df_master.apply(clasificacion_final_detallada, axis=1)

# --- Paso 9: GENERAR DATAFRAME RESUMEN SOLICITADO ---
print("\n" + "="*80)
print("📊 DATAFRAME RESUMEN: CONTEO POR COMBINACIÓN ABC-XYZ-TIPO")
print("="*80)

# Crear DataFrame de resumen con todas las combinaciones
resumen = df_master.groupby(['ABC', 'XYZ', 'Tipo_Demanda', 'CV2_Clasificacion']).size().reset_index(name='Cantidad')

# Crear columna de clasificación simplificada
resumen['Clasificacion'] = resumen.apply(
    lambda row: f"{row['ABC']}{row['XYZ']}-" +
    (f"{row['Tipo_Demanda']}" if row['Tipo_Demanda'] == 'Continua'
     else f"Intermitente ({row['CV2_Clasificacion']})"),
    axis=1
)

# Ordenar
resumen = resumen.sort_values(['ABC', 'XYZ', 'Tipo_Demanda'])

print("\n📋 RESUMEN DETALLADO:")
print(resumen.to_string(index=False))

# Crear DataFrame resumen simplificado
print("\n" + "="*80)
print("📊 RESUMEN SIMPLIFICADO (Agrupado)")
print("="*80)

# Agrupar por ABC-XYZ-Tipo (sin distinguir CV2 para continuas)
resumen_simple = df_master.groupby(['ABC', 'XYZ', 'Tipo_Demanda']).agg({
    'Producto': 'count',
    'Ventas_Total': 'sum',
    'CV': 'mean',
    'ADI': 'mean',
    'CV2': 'mean'
}).reset_index()

resumen_simple.columns = ['ABC', 'XYZ', 'Tipo_Demanda', 'Cantidad', 'Ventas_Total', 'CV_Promedio', 'ADI_Promedio', 'CV2_Promedio']

# Crear columna combinada
resumen_simple['Grupo'] = resumen_simple['ABC'] + resumen_simple['XYZ'] + '-' + resumen_simple['Tipo_Demanda']

# Redondear valores
resumen_simple['CV_Promedio'] = resumen_simple['CV_Promedio'].round(4)
resumen_simple['ADI_Promedio'] = resumen_simple['ADI_Promedio'].round(3)
resumen_simple['CV2_Promedio'] = resumen_simple['CV2_Promedio'].round(4)
resumen_simple['Ventas_Total'] = resumen_simple['Ventas_Total'].round(0)

print("\n" + resumen_simple.to_string(index=False))

# --- Paso 10: Resumen por combinación ABC-XYZ-TIPO ---
print("\n" + "="*80)
print("📊 TABLA FINAL: CANTIDAD POR GRUPO")
print("="*80)

# Crear tabla pivote
tabla_final = resumen_simple[['Grupo', 'Cantidad', 'CV_Promedio', 'ADI_Promedio']].copy()
tabla_final = tabla_final.sort_values('Cantidad', ascending=False)

print("\n" + tabla_final.to_string(index=False))

# Calcular totales y porcentajes
total_items = len(df_master)

print("\n" + "="*80)
print("📊 DISTRIBUCIÓN PORCENTUAL")
print("="*80)

tabla_pct = tabla_final.copy()
tabla_pct['Porcentaje'] = (tabla_pct['Cantidad'] / total_items * 100).round(2)
tabla_pct['Acumulado'] = tabla_pct['Porcentaje'].cumsum().round(2)

print("\n" + tabla_pct.to_string(index=False))

# --- Paso 11: Análisis especial de intermitentes ---
if len(items_intermitentes) > 0:
    print("\n" + "="*80)
    print("🚨 ANÁLISIS DETALLADO DE ITEMS INTERMITENTES")
    print("="*80)

    # Resumen por ABC para intermitentes
    intermitentes_abc = items_intermitentes.groupby(['ABC', 'CV2_Clasificacion']).size().reset_index(name='Cantidad')
    print("\n📊 Items Intermitentes por clase ABC y CV²:")
    print(intermitentes_abc.to_string(index=False))

    # Estadísticas por tipo
    print("\n📊 Estadísticas de Items Intermitentes:")

    suaves = items_intermitentes[items_intermitentes['CV2_Clasificacion'] == 'Suave (Forecasteable)']
    erraticas = items_intermitentes[items_intermitentes['CV2_Clasificacion'] == 'Errática (Difícil)']

    if len(suaves) > 0:
        print(f"\n 🟢 Intermitentes SUAVES (CV² < 0.49): {len(suaves)} items")
        print(f" • Representan: {len(suaves)/len(items_intermitentes)*100:.1f}% de intermitentes")
        print(f" • CV² promedio: {suaves['CV2'].mean():.4f}")
        print(f" • ADI promedio: {suaves['ADI'].mean():.3f}")
        print(f" • Método sugerido: Croston, SBA, TSB")

    if len(erraticas) > 0:
        print(f"\n 🔴 Intermitentes ERRÁTICAS (CV² >= 0.49): {len(erraticas)} items")
        print(f" • Representan: {len(erraticas)/len(items_intermitentes)*100:.1f}% de intermitentes")
        print(f" • CV² promedio: {erraticas['CV2'].mean():.4f}")
        print(f" • ADI promedio: {erraticas['ADI'].mean():.3f}")
        print(f" • Método sugerido: Bootstrap, Simulación, Gestión por excepción")

# --- Paso 12: Matriz de decisión ---
print("\n" + "="*80)
print("🎯 MATRIZ DE DECISIÓN Y ESTRATEGIAS")
print("="*80)

matriz_decision = []

for _, row in resumen_simple.iterrows():
    grupo = row['Grupo']
    cant = row['Cantidad']
    abc = row['ABC']
    xyz = row['XYZ']
    tipo = row['Tipo_Demanda']

    # Determinar prioridad
    if abc == 'A':
        prioridad = 'ALTA'
    elif abc == 'B':
        prioridad = 'MEDIA'
    else:
        prioridad = 'BAJA'

    # Determinar método de pronóstico
    if tipo == 'Continua':
        if xyz == 'X':
            metodo = 'Promedio Móvil, SES'
        elif xyz == 'Y':
            metodo = 'Holt, Winter\'s'
        else: # Z
            metodo = 'ARIMA, ML'
    else: # Intermitente
        cv2_prom = row['CV2_Promedio']
        if pd.notna(cv2_prom) and cv2_prom < 0.49:
            metodo = 'Croston, SBA'
        else:
            metodo = 'TSB, Bootstrap'

    # Determinar estrategia de inventario
    if tipo == 'Continua':
        if xyz == 'X':
            inventario = 'Stock bajo, EOQ'
        elif xyz == 'Y':
            inventario = 'Stock medio, (s,Q)'
        else:
            inventario = 'Stock alto, revisión frecuente'
    else:
        if abc == 'A':
            inventario = 'Stock estratégico, monitoreo diario'
        else:
            inventario = 'Evaluar bajo pedido'

    matriz_decision.append({
        'Grupo': grupo,
        'Cantidad': cant,
        'Prioridad': prioridad,
        'Método_Pronóstico': metodo,
        'Estrategia_Inventario': inventario
    })

df_decision = pd.DataFrame(matriz_decision)
print("\n" + df_decision.to_string(index=False))

# --- Paso 13: Resumen ejecutivo ---
print("\n" + "="*80)
print("📊 RESUMEN EJECUTIVO")
print("="*80)

total_continua = len(df_master[df_master['Tipo_Demanda'] == 'Continua'])
total_intermitente = len(df_master[df_master['Tipo_Demanda'] == 'Intermitente'])

print(f"""
Total de items analizados: {len(df_master)}

CLASIFICACIÓN ABC:
• Clase A: {len(df_master[df_master['ABC'] == 'A'])} items ({len(df_master[df_master['ABC'] == 'A'])/len(df_master)*100:.1f}%)
• Clase B: {len(df_master[df_master['ABC'] == 'B'])} items ({len(df_master[df_master['ABC'] == 'B'])/len(df_master)*100:.1f}%)
• Clase C: {len(df_master[df_master['ABC'] == 'C'])} items ({len(df_master[df_master['ABC'] == 'C'])/len(df_master)*100:.1f}%)

CLASIFICACIÓN XYZ:
• Clase X: {len(df_master[df_master['XYZ'] == 'X'])} items ({len(df_master[df_master['XYZ'] == 'X'])/len(df_master)*100:.1f}%)
• Clase Y: {len(df_master[df_master['XYZ'] == 'Y'])} items ({len(df_master[df_master['XYZ'] == 'Y'])/len(df_master)*100:.1f}%)
• Clase Z: {len(df_master[df_master['XYZ'] == 'Z'])} items ({len(df_master[df_master['XYZ'] == 'Z'])/len(df_master)*100:.1f}%)

TIPO DE DEMANDA:
• Continua: {total_continua} items ({total_continua/len(df_master)*100:.1f}%)
• Intermitente: {total_intermitente} items ({total_intermitente/len(df_master)*100:.1f}%)
""")

if total_intermitente > 0:
    suaves_count = len(df_master[df_master['CV2_Clasificacion'] == 'Suave (Forecasteable)'])
    erraticas_count = len(df_master[df_master['CV2_Clasificacion'] == 'Errática (Difícil)'])

    print(f"""
ITEMS INTERMITENTES (Detalle):
• Suaves (CV² < 0.49): {suaves_count} items ({suaves_count/total_intermitente*100:.1f}% de intermitentes)
→ Forecasteables con Croston/SBA
• Erráticas (CV² >= 0.49): {erraticas_count} items ({erraticas_count/total_intermitente*100:.1f}% de intermitentes)
→ Muy difíciles, requieren métodos especiales
""")

# Items críticos (A-Intermitente)
criticos = df_master[(df_master['ABC'] == 'A') & (df_master['Tipo_Demanda'] == 'Intermitente')]
if len(criticos) > 0:
    print(f"""
⚠️ ITEMS CRÍTICOS (A-Intermitente): {len(criticos)} items
→ Requieren ATENCIÓN ESPECIAL
→ Alta importancia + Alta complejidad
""")
    print(" Productos críticos:")
    for producto in criticos['Producto'].values[:10]:
        print(f" • {producto}")
    if len(criticos) > 10:
        print(f" ... y {len(criticos)-10} más")

print("\n" + "="*80)
print("✅ ANÁLISIS COMPLETADO")
print("="*80)

# Retornar DataFrames principales
print("\n💡 DataFrames generados en memoria:")
print(" • df_master: Clasificación completa de todos los items")
print(" • resumen_simple: Resumen por grupo ABC-XYZ-Tipo")
print(" • tabla_pct: Tabla con porcentajes")
print(" • df_decision: Matriz de decisión con estrategias")

CLASIFICACIÓN COMPLETA: ABC-XYZ-ADI-CV²

✅ Periodo: 2023-01-01 00:00:00 hasta 2025-10-01 00:00:00 (34 períodos)
📊 Productos analizados: 79

PASO 1: CLASIFICACIÓN ABC

📊 Clasificación ABC (80-15-5):
A    61
B    13
C     5
Name: count, dtype: int64

PASO 2: CLASIFICACIÓN XYZ (Coeficiente de Variación)

📊 Clasificación XYZ:
X    26
Y    27
Z    26
Name: count, dtype: int64
 P33 = 0.5747
 P67 = 0.6113

PASO 3: CÁLCULO DE ADI (Average Demand Interval) para items Z

📊 Tipo de Demanda:
Tipo_Demanda
Continua        75
Intermitente     4
Name: count, dtype: int64

PASO 4: CÁLCULO DE CV² para items INTERMITENTES

📖 Clasificación según CV² (Syntetos et al.):
• CV² < 0.49: Suave (Smooth) - Forecasteable con Croston/SBA
• CV² >= 0.49: Errática (Erratic) - Muy difícil de forecastear


📊 Total items intermitentes: 4

 Estadísticas de CV²:
 • Media: 0.8275
 • Mediana: 0.8002
 • Mínimo: 0.7734
 • Máximo: 0.9363

📊 Clasificación por CV²:
CV2_Clasificacion
Errática (Difícil)    4
Name: count, dtype: int